# Bayesian Inference — Exercises

Companion to the note [Bayesian Inference](Bayesian%20Inference.md).

Practise Bayes' rule, conjugate updates (Beta–Bernoulli, Gaussian–Gaussian, Dirichlet–Multinomial), MAP vs posterior mean, posterior predictive distributions, credible intervals, Bayesian linear regression (MAP = ridge) and model comparison with the evidence, by hand and in NumPy/SciPy.

**15 exercises** · 🧠 Concept ×3 · ✍️ By hand ×7 · 💻 Code ×5

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy import stats
from scipy.special import betaln, gammaln
from sklearn.linear_model import Ridge
rng = np.random.default_rng(0)
# Shared coin data: 7 heads in 10 tosses
coin = np.array([1, 1, 0, 1, 1, 0, 1, 1, 0, 1])

## Part A · Concepts

### Exercise 1 · The four terms of Bayes' rule
*🧠 Concept · ★☆☆*

Name and interpret each term of $p(\theta\mid D)=\frac{p(D\mid\theta)p(\theta)}{p(D)}$. Which of them is a probability distribution over $\theta$?
Why can we often ignore $p(D)$ when computing the posterior, and when can we **not** ignore it?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

As a function of $\theta$, the likelihood is not normalized. $p(D)$ does not depend on $\theta$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

- **Prior** $p(\theta)$: belief before seeing data (a distribution over $\theta$).
- **Likelihood** $p(D\mid\theta)$: how probable the data is for each $\theta$; a function of $\theta$, *not* a distribution over it (it need not integrate to 1).
- **Evidence** $p(D)=\int p(D\mid\theta)p(\theta)d\theta$: the normalizing constant; a single number for a given model.
- **Posterior** $p(\theta\mid D)$: updated belief, a distribution over $\theta$.
For the *shape* of the posterior $p(\theta\mid D)\propto p(D\mid\theta)p(\theta)$ is enough (MAP, MCMC, grid normalization). We need $p(D)$ for **model comparison** (it is $p(D\mid M)$), and it is precisely what makes posteriors intractable ([[Variational Inference]], [[MCMC]]).

</details>

### Exercise 2 · Plug-in vs posterior predictive
*🧠 Concept · ★☆☆*

A coin lands heads 3 times in 3 tosses. Compare the probability of heads on the next toss under (a) the MLE plug-in and
(b) the posterior predictive with a uniform $\mathrm{Beta}(1,1)$ prior. Which is more sensible and why?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Beta–Bernoulli: $p(y_*=1\mid D)=\frac{a+h}{a+b+N}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $\hat\theta_{MLE}=3/3=1$, so the plug-in says tails is **impossible**, which is absurd after 3 tosses.
(b) Posterior $\mathrm{Beta}(4,1)$, predictive $p=4/5=0.8$ (Laplace's rule of succession).
The predictive averages over all $\theta$ compatible with the data instead of betting on a single point estimate, so it stays appropriately uncertain with little data. With lots of data both agree.

</details>

### Exercise 3 · Automatic Occam's razor
*🧠 Concept · ★★☆*

The note says model comparison via the evidence $p(D\mid M)$ is an "automatic Occam's razor". Explain why a very flexible model can have
**lower** evidence than a simple one, even though its best-fitting parameters fit the data better.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$p(D\mid M)$ is a distribution over data sets: it must sum to 1 over all possible $D$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$p(D\mid M)=\int p(D\mid\theta,M)p(\theta\mid M)d\theta$ is a probability distribution over all possible data sets. A flexible model can explain many different data sets, so it must spread its probability mass thinly; a simple model concentrates its mass on few data sets.
If the observed data is one the simple model predicts well, the simple model gets higher evidence. Equivalently, the evidence is (best-fit likelihood) × (Occam factor $\approx$ posterior width / prior width), and flexible models pay for the prior volume they waste. Exercise 10 and Exercise 15 show it numerically. BIC is a large-sample approximation of $-2\log p(D\mid M)$.

</details>

## Part B · By hand

### Exercise 4 · A positive test
*✍️ By hand · ★☆☆*

A disease has prevalence 1%. A test has sensitivity $P(+\mid D)=0.99$ and specificity $P(-\mid\neg D)=0.95$. You test positive.
What is $P(D\mid +)$? What is it after a **second**, independent positive test?

In [ ]:
p_one = None
p_two = None

_post = lambda prior: 0.99 * prior / (0.99 * prior + 0.05 * (1 - prior))
check('P(D | +)', p_one, _post(0.01))
check('P(D | +, +)', p_two, _post(_post(0.01)))

<details>
<summary><b>💡 Hint</b></summary>

Today's posterior is tomorrow's prior.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$P(D\mid+)=\frac{0.99\cdot0.01}{0.99\cdot0.01+0.05\cdot0.99}=\frac{0.0099}{0.0594}=\frac16\approx0.167$.
Second test with prior $1/6$: $\frac{0.99/6}{0.99/6+0.05\cdot5/6}=\frac{0.99}{0.99+0.25}\approx0.798$.
The low prior (base rate) dominates the first result; only repeated evidence makes the disease likely.

```python
p_one = 1/6
p_two = 0.99 / (0.99 + 0.25)
```

</details>

### Exercise 5 · Beta–Bernoulli update
*✍️ By hand · ★☆☆*

Prior $\theta\sim\mathrm{Beta}(2,2)$, data `coin` (7 heads, 3 tails). Give the posterior parameters $(a_N,b_N)$,
the posterior mean, the MAP estimate and the MLE. Order them and explain.

In [ ]:
aN = None
bN = None
post_mean = None
map_est = None
mle = None

_h = coin.sum(); _t = len(coin) - _h
check('a_N', aN, 2 + _h); check('b_N', bN, 2 + _t)
check('posterior mean', post_mean, stats.beta(2 + _h, 2 + _t).mean())
_g = np.linspace(0, 1, 200001); check('MAP (grid)', map_est, _g[np.argmax(stats.beta(2 + _h, 2 + _t).pdf(_g))], tol=1e-4)
check('MLE', mle, coin.mean())

<details>
<summary><b>💡 Hint</b></summary>

Posterior $\mathrm{Beta}(a+h,\,b+t)$; mean $\frac{a}{a+b}$; mode $\frac{a-1}{a+b-2}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\mathrm{Beta}(9,5)$. Mean $9/14\approx0.643$, MAP $8/12\approx0.667$, MLE $0.7$.
The prior $\mathrm{Beta}(2,2)$ acts like 1 extra head and 1 extra tail (for the MAP) or 2 and 2 (for the mean), pulling estimates towards $0.5$. With more data all three converge.

```python
aN, bN = 9, 5
post_mean = 9/14
map_est = 8/12
mle = 0.7
```

</details>

### Exercise 6 · Posterior predictive for two tosses
*✍️ By hand · ★★☆*

With the $\mathrm{Beta}(9,5)$ posterior, compute (a) $p(\text{next toss}=H\mid D)$ and (b) $p(\text{next two tosses}=HH\mid D)$.
Why is (b) **not** the square of (a)?

In [ ]:
p_next_H = None
p_next_HH = None

check('P(H | D)', p_next_H, np.exp(betaln(10, 5) - betaln(9, 5)))
check('P(HH | D)', p_next_HH, np.exp(betaln(11, 5) - betaln(9, 5)))

<details>
<summary><b>💡 Hint</b></summary>

$p(HH\mid D)=\mathbb E[\theta^2]=\mathbb E[\theta]^2+\mathrm{Var}[\theta]$, or $\frac{a(a+1)}{(a+b)(a+b+1)}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $\mathbb E[\theta]=9/14\approx0.643$. (b) $\mathbb E[\theta^2]=\frac{9\cdot10}{14\cdot15}=\frac{90}{210}\approx0.429$, while $0.643^2\approx0.413$.
The tosses are independent *given* $\theta$, but not marginally: a head on the first toss makes high $\theta$ more plausible. The difference is exactly $\mathrm{Var}[\theta\mid D]$, the posterior uncertainty.

```python
p_next_H = 9/14
p_next_HH = 90/210
```

</details>

### Exercise 7 · Gaussian–Gaussian: mean with known variance
*✍️ By hand · ★★☆*

Observations $x_i\sim\mathcal N(\mu,\sigma^2)$ with known $\sigma^2=4$; prior $\mu\sim\mathcal N(\mu_0=0,\ \tau_0^2=1)$. You observe $N=4$ points with mean $\bar x=3$.
Compute the posterior mean and variance of $\mu$. Interpret the posterior mean as a weighted average.

In [ ]:
mu_post = None
var_post = None

_prec = 1 / 1 + 4 / 4
check('posterior variance', var_post, 1 / _prec)
check('posterior mean', mu_post, (0 / 1 + 4 * 3 / 4) / _prec)

<details>
<summary><b>💡 Hint</b></summary>

Precisions add: $\frac1{\tau_N^2}=\frac1{\tau_0^2}+\frac N{\sigma^2}$, and $\mu_N=\tau_N^2\left(\frac{\mu_0}{\tau_0^2}+\frac{N\bar x}{\sigma^2}\right)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Precision $1+4/4=2\Rightarrow\tau_N^2=0.5$; $\mu_N=0.5\,(0+3)=1.5$.
$\mu_N=\frac{\lambda_0}{\lambda_0+\lambda_D}\mu_0+\frac{\lambda_D}{\lambda_0+\lambda_D}\bar x$ with prior precision $\lambda_0=1$ and data precision $\lambda_D=N/\sigma^2=1$: here prior and data get equal weight, so the estimate is halfway between 0 and 3.

```python
mu_post = 1.5
var_post = 0.5
```

</details>

### Exercise 8 · Dirichlet–Multinomial and add-one smoothing
*✍️ By hand · ★☆☆*

A three-sided die: prior $\mathrm{Dir}(1,1,1)$, observed counts $(5,0,3)$. Give the posterior and its mean. Compare with the MLE and
explain the link to Laplace smoothing in [[Naive Bayes]].

In [ ]:
dir_post_mean = None  # 3 numbers

check('posterior mean', dir_post_mean, stats.dirichlet(np.array([1, 1, 1]) + [5, 0, 3]).mean())

<details>
<summary><b>💡 Hint</b></summary>

Posterior $\mathrm{Dir}(\alpha+n)$, mean $\frac{\alpha_k+n_k}{\sum_j(\alpha_j+n_j)}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\mathrm{Dir}(6,1,4)$ with mean $(6,1,4)/11\approx(0.545,0.091,0.364)$. The MLE $(5/8,0,3/8)$ declares face 2 impossible.
The posterior mean with a uniform Dirichlet prior is exactly **add-one (Laplace) smoothing**, used in Naive Bayes so that an unseen word does not zero out a whole class probability.

```python
dir_post_mean = np.array([6, 1, 4]) / 11
```

</details>

### Exercise 9 · MAP for linear regression = ridge
*✍️ By hand · ★★☆*

With $y_i=w^\top x_i+\varepsilon_i$, $\varepsilon_i\sim\mathcal N(0,\beta^{-1})$ and the note's prior $w\sim\mathcal N(0,\alpha^{-1}I)$,
show that the MAP estimate minimizes the ridge objective $\lVert Xw-y\rVert^2+\lambda\lVert w\rVert^2$ and find $\lambda$.
Also give the full Gaussian posterior $\mathcal N(m_N,S_N)$. See [[Linear Regression]].

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Take $-\log$ of likelihood × prior and drop constants. Complete the square for the posterior.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$-\log p(w\mid D)=\frac\beta2\lVert Xw-y\rVert^2+\frac\alpha2\lVert w\rVert^2+\text{const}$. Multiplying by $2/\beta$: ridge with $\lambda=\alpha/\beta$.
The exponent is quadratic in $w$, so the posterior is Gaussian with
$S_N^{-1}=\alpha I+\beta X^\top X$ and $m_N=\beta S_NX^\top y=(X^\top X+\tfrac\alpha\beta I)^{-1}X^\top y$, which is the ridge solution (mean = mode for a Gaussian).
Bayesian regression additionally provides $S_N$, the uncertainty in $w$, which ridge discards.

</details>

### Exercise 10 · Is the coin fair? A Bayes factor
*✍️ By hand · ★★★*

A specific sequence of 10 tosses contains 9 heads. Compare $M_0$: $\theta=0.5$ (fair) with $M_1$: $\theta\sim\mathrm{Beta}(1,1)$.
Compute both evidences $p(D\mid M_0)$, $p(D\mid M_1)$ and the Bayes factor $K=p(D\mid M_1)/p(D\mid M_0)$.

In [ ]:
ev_M0 = None
ev_M1 = None
bayes_factor = None

from scipy.integrate import quad
_ev1 = quad(lambda t: t ** 9 * (1 - t), 0, 1)[0]
check('p(D | M0)', ev_M0, 0.5 ** 10, tol=1e-9)
check('p(D | M1) (numerical integral)', ev_M1, _ev1, tol=1e-9)
check('Bayes factor', bayes_factor, _ev1 / 0.5 ** 10)

<details>
<summary><b>💡 Hint</b></summary>

$p(D\mid M_1)=\int_0^1\theta^h(1-\theta)^t\,d\theta=B(h+1,t+1)=\frac{h!\,t!}{(h+t+1)!}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$p(D\mid M_0)=2^{-10}=1/1024\approx9.77\cdot10^{-4}$. $p(D\mid M_1)=B(10,2)=\frac{9!\,1!}{11!}=\frac1{110}\approx9.09\cdot10^{-3}$.
$K=1024/110\approx9.3$: "substantial" evidence for a biased coin (Jeffreys' scale), but not overwhelming. With 5 heads in 10, $M_0$ would win ($K\approx0.37$), because $M_1$ wastes prior mass on extreme $\theta$: Occam's razor at work.

```python
ev_M0 = 1 / 1024
ev_M1 = 1 / 110
bayes_factor = 1024 / 110
```

</details>

## Part C · Code

### Exercise 11 · Grid approximation of a posterior
*💻 Code · ★☆☆*

Compute the posterior of $\theta$ for `coin` on a grid of 2001 points in $[0,1]$: evaluate prior × likelihood, normalize so it integrates to 1
(trapezoid rule or simple sum × spacing). Do it for (a) the $\mathrm{Beta}(2,2)$ prior and (b) a **non-conjugate** prior
$p(\theta)\propto\exp(-(\theta-0.5)^2/(2\cdot0.05^2))$ (a strong belief in fairness). Store both posterior means.

In [ ]:
grid = np.linspace(0, 1, 2001)
mean_beta_prior = None
mean_gauss_prior = None

check('posterior mean, Beta(2,2) prior', mean_beta_prior, 9 / 14, tol=1e-4)
_s = stats.truncnorm((0 - .5) / .05, (1 - .5) / .05, .5, .05).rvs(400000, random_state=1)
_w = _s ** 7 * (1 - _s) ** 3
check('posterior mean, Gaussian prior (importance sampling)', mean_gauss_prior, np.sum(_w * _s) / _w.sum(), tol=2e-3)

<details>
<summary><b>💡 Hint</b></summary>

Work with log-likelihood $h\log\theta+t\log(1-\theta)$ to avoid underflow with more data; the grid end points give $\log 0$, which is fine after `exp` (or drop them).

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $\approx0.6429=9/14$, matching the conjugate result. (b) $\approx0.52$: the strong prior keeps the estimate near 0.5 despite 7/10 heads.
Grid approximation works for any prior in 1–2 dimensions, but the number of grid points grows exponentially with dimension: that is why we need [[MCMC]] or [[Variational Inference]].

```python
grid = np.linspace(0, 1, 2001)
h, t = coin.sum(), len(coin) - coin.sum()
with np.errstate(divide='ignore'):
    loglik = h * np.log(grid) + t * np.log(1 - grid)
def post_mean(log_prior):
    lp = loglik + log_prior
    p = np.exp(lp - lp.max())
    p /= p.sum()
    return np.sum(grid * p)
with np.errstate(divide='ignore'):
    mean_beta_prior = post_mean(np.log(grid) + np.log(1 - grid))
mean_gauss_prior = post_mean(-(grid - 0.5) ** 2 / (2 * 0.05 ** 2))
print(mean_beta_prior, mean_gauss_prior)
```

</details>

### Exercise 12 · Credible intervals: equal-tailed vs HDI
*💻 Code · ★★☆*

For the posterior $\mathrm{Beta}(9,5)$ compute (a) the 95% **equal-tailed** interval with `stats.beta.ppf` and (b) the 95% **highest-density interval**
(HDI): the shortest interval containing 95% mass. For (b), search over the lower tail probability $p\in[0,0.05]$ for the interval
$[F^{-1}(p),F^{-1}(p+0.95)]$ of minimal width. Store `ci_eq` and `ci_hdi` as pairs.

In [ ]:
ci_eq = None
ci_hdi = None

_b = stats.beta(9, 5)
check('equal-tailed interval', ci_eq, _b.ppf([0.025, 0.975]))
check('HDI: 95% mass, equal density at both ends, shorter', None if ci_hdi is None else
      (abs(_b.cdf(ci_hdi[1]) - _b.cdf(ci_hdi[0]) - 0.95) < 1e-3 and abs(_b.pdf(ci_hdi[0]) - _b.pdf(ci_hdi[1])) < 2e-2
       and ci_hdi[1] - ci_hdi[0] < ci_eq[1] - ci_eq[0]), True)

<details>
<summary><b>💡 Hint</b></summary>

`ps = np.linspace(0, 0.05, 5001)`; widths `b.ppf(ps + 0.95) - b.ppf(ps)`; take the argmin.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Equal-tailed $\approx(0.386,0.861)$; HDI $\approx(0.401,0.874)$, slightly shorter and shifted, because the posterior is skewed. At the HDI end points the density is equal.
Interpretation (unlike a frequentist confidence interval): given the model and prior, $\theta$ lies in the interval with 95% probability.

```python
_b = stats.beta(9, 5)
ci_eq = _b.ppf([0.025, 0.975])
ps = np.linspace(0, 0.05, 5001)
widths = _b.ppf(ps + 0.95) - _b.ppf(ps)
p_best = ps[np.argmin(widths)]
ci_hdi = (_b.ppf(p_best), _b.ppf(p_best + 0.95))
print(np.round(ci_eq, 3), np.round(ci_hdi, 3))
```

</details>

### Exercise 13 · Bayesian linear regression posterior
*💻 Code · ★★☆*

Implement `blr_posterior(X, y, alpha, beta)` returning $(m_N,S_N)$ from Exercise 9 (no intercept). On the data below with $\alpha=2$, $\beta=25$
check that $m_N$ equals `Ridge(alpha=alpha/beta, fit_intercept=False)`. Then draw 5 weight samples from the posterior.

In [ ]:
X_b = rng.normal(size=(30, 2))
y_b = X_b @ np.array([1.0, -0.5]) + rng.normal(scale=0.2, size=30)
def blr_posterior(X, y, alpha, beta):
    # TODO
    return None, None

mN, SN = blr_posterior(X_b, y_b, 2.0, 25.0)

check('posterior mean = ridge', mN, Ridge(alpha=2.0 / 25.0, fit_intercept=False).fit(X_b, y_b).coef_)
check('posterior covariance is symmetric PD', None if SN is None else bool(np.allclose(SN, SN.T) and np.all(np.linalg.eigvalsh(SN) > 0)), True)

<details>
<summary><b>💡 Hint</b></summary>

`SN = np.linalg.inv(alpha * np.eye(d) + beta * X.T @ X)`; `mN = beta * SN @ X.T @ y`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$m_N$ matches ridge exactly. The samples scatter around $m_N$ with standard deviations $\sqrt{\mathrm{diag}\,S_N}\approx0.04$: with 30 points and low noise the weights are well determined. With fewer points or more noise, $S_N$ grows.

```python
X_b = rng.normal(size=(30, 2))
y_b = X_b @ np.array([1.0, -0.5]) + rng.normal(scale=0.2, size=30)
def blr_posterior(X, y, alpha, beta):
    SN = np.linalg.inv(alpha * np.eye(X.shape[1]) + beta * X.T @ X)
    return beta * SN @ X.T @ y, SN

mN, SN = blr_posterior(X_b, y_b, 2.0, 25.0)
print(rng.multivariate_normal(mN, SN, size=5))
```

</details>

### Exercise 14 · Predictive uncertainty grows away from the data
*💻 Code · ★★☆*

1-D regression with features $\phi(x)=(1,x)$: training inputs only in $[-1,1]$. Using `blr_posterior` with $\alpha=1$, $\beta=25$, compute the
predictive variance $\sigma^2(x_*)=\frac1\beta+\phi(x_*)^\top S_N\phi(x_*)$ on a grid in $[-4,4]$ and store it in `pred_var`.
Verify it against a Monte Carlo estimate: sample $w\sim\mathcal N(m_N,S_N)$, $y_*=w^\top\phi+\varepsilon$, and take the variance.

In [ ]:
x_tr = rng.uniform(-1, 1, 20)
y_tr = 0.5 + 2 * x_tr + rng.normal(scale=0.2, size=20)
x_grid = np.linspace(-4, 4, 9)
Phi_tr = np.c_[np.ones(20), x_tr]; Phi_grid = np.c_[np.ones(9), x_grid]
pred_var = None

_m, _S = np.linalg.solve(np.eye(2) + 25 * Phi_tr.T @ Phi_tr, 25 * Phi_tr.T @ y_tr), np.linalg.inv(np.eye(2) + 25 * Phi_tr.T @ Phi_tr)
_r = np.random.default_rng(9); _W = _r.multivariate_normal(_m, _S, 200000)
_mc = (_W @ Phi_grid.T + _r.normal(scale=0.2, size=(200000, 9))).var(0)
check('predictive variance vs Monte Carlo', pred_var, _mc, tol=3e-3)
check('larger far from data', None if pred_var is None else pred_var[0] > 2 * pred_var[4], True)

<details>
<summary><b>💡 Hint</b></summary>

`np.einsum('ij,jk,ik->i', Phi, SN, Phi)` computes $\phi_i^\top S_N\phi_i$ for every row.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The noise term $1/\beta=0.04$ is constant; the parameter-uncertainty term $\phi^\top S_N\phi$ is smallest near the training inputs and grows quadratically outside $[-1,1]$.
A plug-in prediction with $m_N$ alone would report the same $\pm0.2$ everywhere, overconfident when extrapolating. [[Gaussian Processes]] generalize this behaviour.

```python
x_tr = rng.uniform(-1, 1, 20)
y_tr = 0.5 + 2 * x_tr + rng.normal(scale=0.2, size=20)
x_grid = np.linspace(-4, 4, 9)
Phi_tr = np.c_[np.ones(20), x_tr]; Phi_grid = np.c_[np.ones(9), x_grid]
m1, S1 = blr_posterior(Phi_tr, y_tr, 1.0, 25.0)
pred_var = 1 / 25 + np.einsum('ij,jk,ik->i', Phi_grid, S1, Phi_grid)
print(np.round(pred_var, 4))
```

</details>

### Exercise 15 · Evidence-based choice of polynomial degree
*💻 Code · ★★★*

For Bayesian polynomial regression with fixed $\alpha,\beta$, the log evidence is (Bishop eq. 3.86)
$$\log p(y\mid\alpha,\beta)=\tfrac M2\log\alpha+\tfrac N2\log\beta-E(m_N)-\tfrac12\log|A|-\tfrac N2\log2\pi,$$
with $M$ = number of basis functions, $A=S_N^{-1}$, $E(m)=\frac\beta2\lVert y-\Phi m\rVert^2+\frac\alpha2 m^\top m$.
Implement `log_evidence(Phi, y, alpha, beta)` and evaluate it for polynomial degrees $0..8$ on the data below ($\alpha=5\cdot10^{-3}$, $\beta=11.1$).
Store the values in `log_ev` and the best degree in `best_degree`.

In [ ]:
r6 = np.random.default_rng(2)
x_p = np.linspace(0, 1, 10)
y_p = np.sin(2 * np.pi * x_p) + r6.normal(scale=0.3, size=10)
def log_evidence(Phi, y, alpha, beta):
    # TODO
    return None

log_ev = None
best_degree = None

_ref = [stats.multivariate_normal(np.zeros(10), np.eye(10) / 11.1 + np.vander(x_p, d + 1, increasing=True) @ np.vander(x_p, d + 1, increasing=True).T / 5e-3).logpdf(y_p) for d in range(9)]
check('log evidence (vs marginal Gaussian y ~ N(0, I/beta + Phi Phi^T/alpha))', log_ev, _ref, tol=1e-6)
check('best degree', best_degree, int(np.argmax(_ref)))

<details>
<summary><b>💡 Hint 1</b></summary>

Basis: `np.vander(x, M, increasing=True)` gives $1,x,\dots,x^{M-1}$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Use `np.linalg.slogdet(A)[1]` for $\log|A|$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The check uses an independent route: integrating out $w$ gives $y\sim\mathcal N(0,\beta^{-1}I+\alpha^{-1}\Phi\Phi^\top)$ directly.
The evidence is poor for degrees 0–2, peaks at degree 3–4 (degree 4 for this noise draw, with 3 close behind), then falls slowly: higher degrees fit the training data at least as well, but pay an Occam penalty for their larger prior volume. Unlike training error, the evidence does not keep improving with complexity, and no validation set was needed.

```python
r6 = np.random.default_rng(2)
x_p = np.linspace(0, 1, 10)
y_p = np.sin(2 * np.pi * x_p) + r6.normal(scale=0.3, size=10)
def log_evidence(Phi, y, alpha, beta):
    N, M = Phi.shape
    A = alpha * np.eye(M) + beta * Phi.T @ Phi
    m = beta * np.linalg.solve(A, Phi.T @ y)
    E = beta / 2 * np.sum((y - Phi @ m) ** 2) + alpha / 2 * m @ m
    return M / 2 * np.log(alpha) + N / 2 * np.log(beta) - E - 0.5 * np.linalg.slogdet(A)[1] - N / 2 * np.log(2 * np.pi)

log_ev = [log_evidence(np.vander(x_p, d + 1, increasing=True), y_p, 5e-3, 11.1) for d in range(9)]
best_degree = int(np.argmax(log_ev))
print(np.round(log_ev, 2), best_degree)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Bayesian Inference](Bayesian%20Inference.md).*